<a href="https://colab.research.google.com/github/Agustinadearagon/rastro/blob/main/rastro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title 🔎 Comprueba tu huella digital
#@markdown **Rellena los campos que quieras** (puedes dejar alguno vacío) y pulsa ▶.
#@markdown Después espera unos minutos: al terminar se descargará tu informe.
username = ""  #@param {type:"string"}
email = ""  #@param {type:"string"}
#@markdown ---
#@markdown Esta herramienta es solo para revisar **tus propios datos**.
confirmo = False  #@param {type:"boolean"}
#@markdown ☝️ Marca la casilla si los datos son tuyos o tienes permiso de su dueño.

import subprocess, os, re, html, datetime, shutil, glob, json

username = username.strip()
email = email.strip()

if not confirmo:
    raise SystemExit("⚠️ Marca la casilla de confirmación y vuelve a pulsar ▶")
if not (username or email):
    raise SystemExit("⚠️ Rellena al menos un campo y vuelve a pulsar ▶")
if email and not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
    raise SystemExit("⚠️ El correo no parece válido")
if username and not re.fullmatch(r"[A-Za-z0-9._-]+", username):
    raise SystemExit("⚠️ El usuario solo puede llevar letras, números, punto, guion y guion bajo")

CARPETA = "/content/reports"
BB = "/content/blackbird"
shutil.rmtree(CARPETA, ignore_errors=True)
os.makedirs(CARPETA)
os.chdir("/content")

def ejecutar(comando, limite=900, cwd=None):
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True, timeout=limite, cwd=cwd)
        salida = r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        salida = "La herramienta tardó demasiado y se detuvo."
    return salida.strip()

# ---------- Limpieza de resultados ----------
RUIDO = [
    "OSINTSearch", "osintsearch.org", "Go deeper than a username",
    "Donate on Patreon", "Support Maigret", "DB auto-update",
    "could not be decoded", "Searching |", "it/s]",
    "You can run search by full list", "You can see detailed site check",
]

# Datos internos que no aportan nada al informe
CLAVES_INTERNAS = {
    "id", "uid", "sec_uid", "tiktok_id", "facebook_uid", "youtube_channel_id",
    "chess_user_id", "status", "is_streamer",
    "is_joined_recently", "is_business", "is_secret", "is_family_safe",
    "digg_count", "description", "image",
}

# Webs para adultos que se quitan del informe
ADULTOS = (
    "chaturbate", "pornhub", "xvideos", "xhamster", "xnxx", "redtube",
    "youporn", "spankbang", "stripchat", "bongacams", "cam4", "camsoda",
    "myfreecams", "livejasmin", "onlyfans", "fansly", "porn", "erotic",
)

# Enlaces de perfil para webs en las que Blackbird da una dirección técnica
PERFILES = {
    "X": "https://x.com/{u}",
    "TikTok": "https://www.tiktok.com/@{u}",
    "Roblox": "https://www.roblox.com/search/users?keyword={u}",
    "GitHub (User)": "https://github.com/{u}",
    "Chess.com": "https://www.chess.com/member/{u}",
}

def es_adulto(texto):
    t = texto.lower()
    return any(x in t for x in ADULTOS)

def limpiar(texto):
    texto = re.sub(r"\x1b\][^\x1b]*\x1b\\", "", texto)
    texto = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", texto)
    salida = []
    for linea in texto.splitlines():
        if not linea.strip():
            continue
        if any(x in linea for x in RUIDO):
            continue
        salida.append(linea.rstrip())
    return "\n".join(salida)

def limpiar_sherlock(t):
    lineas = [l for l in limpiar(t).splitlines()
              if l.startswith("[+]") and not es_adulto(l)]
    return "\n".join(lineas) or "No se encontró ninguna cuenta."

def limpiar_maigret(t):
    salida, resumen, omitir = [], False, False
    for l in limpiar(t).splitlines():
        s = l.strip()
        if "Short text report" in s:
            resumen = True
            salida.append("\nResumen:")
            continue
        if resumen:
            if not s.startswith(("Interests", "Found target's other IDs", "Search by username")):
                salida.append(s)
            continue
        if s.startswith("[+]"):
            if "MAIGRET" in s or "sites database" in s:
                continue
            omitir = es_adulto(s)
            if omitir:
                continue
            salida.append("")
            salida.append(s)
        elif s.startswith(("├─", "└─")):
            if omitir:
                continue
            detalle = s[2:].strip()
            clave = detalle.split(":")[0].strip()
            if clave in CLAVES_INTERNAS:
                continue
            salida.append("   - " + detalle)
    return "\n".join(salida).strip() or "No se encontró ninguna cuenta."

def es_tecnica(url):
    return any(x in url for x in ("api.", "/oembed", "/v1/", "/i/", "/callback/", "/signin"))

def formatear_blackbird(archivos, usuario=None, correo=False):
    lineas = []
    for ruta in archivos:
        try:
            with open(ruta, encoding="utf-8") as f:
                datos = json.load(f)
        except Exception:
            continue
        if isinstance(datos, dict):
            datos = datos.get("results") or datos.get("data") or []
        for d in datos:
            if not isinstance(d, dict) or d.get("status") != "FOUND":
                continue
            nombre = d.get("name", "?")
            url = d.get("url", "")
            if correo:
                url = ""
            elif es_tecnica(url):
                plantilla = PERFILES.get(nombre)
                url = plantilla.format(u=usuario) if (plantilla and usuario) else ""
            linea = f"[+] {nombre}"
            if d.get("category"):
                linea += f" ({d['category']})"
            if url:
                linea += f": {url}"
            lineas.append("")
            lineas.append(linea)
            for m in d.get("metadata") or []:
                if isinstance(m, dict) and m.get("name") and m.get("value") not in (None, ""):
                    clave = str(m["name"]).lower()
                    if any(x in clave for x in ("image", "avatar", "photo", "picture")):
                        continue
                    lineas.append(f"   - {m['name']}: {str(m['value'])[:150]}")
    return "\n".join(lineas).strip() or "No se encontró ninguna cuenta."

def n_cuentas(t):
    return len([l for l in t.splitlines() if l.startswith("[+]")])

secciones, resumen_final, crudo = [], [], ""

# ---------- Sherlock y Maigret ----------
if username:
    print("⏳ Preparando Sherlock y Maigret (1-2 minutos)...")
    ejecutar("pip install -q sherlock-project maigret")

    print("⏳ Buscando el usuario con Sherlock...")
    s = ejecutar(f"sherlock {username} --print-found --timeout 10 --folderoutput {CARPETA}")
    crudo += f"\n===== SHERLOCK =====\n{s}\n"
    s_limpio = limpiar_sherlock(s)
    secciones.append((f"Usuario '{username}' - Sherlock", s_limpio))
    resumen_final.append(f"Sherlock: {n_cuentas(s_limpio)} cuentas")

    print("⏳ Buscando el usuario con Maigret (puede tardar varios minutos)...")
    m = ejecutar(f"maigret {username} --html --timeout 10 --no-progressbar --no-color --no-recursion --folderoutput {CARPETA}")
    crudo += f"\n===== MAIGRET =====\n{m}\n"
    m_limpio = limpiar_maigret(m)
    secciones.append((f"Usuario '{username}' - Maigret", m_limpio))
    resumen_final.append(f"Maigret: {n_cuentas(m_limpio)} cuentas")

# ---------- Blackbird (se instala al final para no interferir) ----------
print("⏳ Preparando Blackbird...")
if not os.path.exists(BB):
    ejecutar(f"git clone -q https://github.com/p1ngul1n0/blackbird.git {BB}")
    ejecutar(f"pip install -q -r {BB}/requirements.txt")

def buscar_blackbird(argumento):
    # Se vacía la carpeta de resultados para no leer búsquedas anteriores
    shutil.rmtree(f"{BB}/results", ignore_errors=True)
    os.makedirs(f"{BB}/results", exist_ok=True)
    salida = ejecutar(f"python blackbird.py {argumento} --no-nsfw --json", limite=900, cwd=BB)
    archivos = sorted(glob.glob(f"{BB}/results/**/*.json", recursive=True))
    return salida, archivos

if username:
    print("⏳ Buscando el usuario con Blackbird...")
    b, archivos = buscar_blackbird(f'-u "{username}"')
    crudo += f"\n===== BLACKBIRD (usuario) =====\n{b}\n"
    b_limpio = formatear_blackbird(archivos, usuario=username)
    secciones.append((f"Usuario '{username}' - Blackbird", b_limpio))
    resumen_final.append(f"Blackbird (usuario): {n_cuentas(b_limpio)} cuentas")

if email:
    print("⏳ Buscando el correo con Blackbird...")
    b, archivos = buscar_blackbird(f'-e "{email}"')
    crudo += f"\n===== BLACKBIRD (correo) =====\n{b}\n"
    b_limpio = formatear_blackbird(archivos, correo=True)
    secciones.append((f"Email '{email}' - Blackbird", b_limpio))
    resumen_final.append(f"Blackbird (correo): {n_cuentas(b_limpio)} cuentas")

with open(f"{CARPETA}/salida_sin_filtrar.txt", "w", encoding="utf-8") as f:
    f.write(crudo)

# ---------- Informe ----------
fecha = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")
resumen_texto = " | ".join(resumen_final)

texto = f"INFORME DE HUELLA DIGITAL\nFecha: {fecha}\n"
if resumen_texto:
    texto += f"Resumen: {resumen_texto}\n"
for titulo, contenido in secciones:
    texto += f"\n{'=' * 60}\n{titulo}\n{'=' * 60}\n{contenido}\n"
with open(f"{CARPETA}/informe_huella_digital.txt", "w", encoding="utf-8") as f:
    f.write(texto)

def enlazar(t):
    t = html.escape(t)
    return re.sub(r"(https?://[^\s<]+)", r'<a href="\1" target="_blank">\1</a>', t)

cuerpo = ""
for titulo, contenido in secciones:
    cuerpo += f"<h2>{html.escape(titulo)}</h2><pre>{enlazar(contenido)}</pre>"

pagina = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Informe de huella digital</title>
<style>
body {{ font-family: sans-serif; max-width: 900px; margin: auto; padding: 16px; }}
h2 {{ background: #eef; padding: 8px; border-radius: 6px; }}
pre {{ white-space: pre-wrap; word-break: break-word; background: #f6f6f6; padding: 12px; border-radius: 6px; }}
.aviso {{ color: #555; font-size: 0.9em; }}
</style></head><body>
<h1>Informe de huella digital</h1>
<p>Fecha: {fecha}</p>
<p><b>{html.escape(resumen_texto)}</b></p>
<p class="aviso">Que exista una cuenta con ese nombre no significa que sea tuya. Algunas webs bloquean la comprobación, así que el resultado puede estar incompleto.</p>
{cuerpo}
</body></html>"""
ruta_html = f"{CARPETA}/informe_huella_digital.html"
with open(ruta_html, "w", encoding="utf-8") as f:
    f.write(pagina)

print("\n✅ Listo")
if resumen_texto:
    print(resumen_texto)
print("Descargando tu informe...")

from google.colab import files
files.download(ruta_html)